In [1]:
!wget -qN -P inputs https://raw.githubusercontent.com/openforcefield/openff-docs/main/source/workshops/2024/vignettes/inputs/PT2385.sdf
!wget -qN -P inputs https://raw.githubusercontent.com/openforcefield/openff-docs/main/source/workshops/2024/vignettes/inputs/solvated_complex.pdb

In [2]:
from openff.toolkit import ForceField, Molecule, Topology

ligand = Molecule.from_file("inputs/PT2385.sdf")
top = Topology.from_pdb("inputs/solvated_complex.pdb", unique_molecules=[ligand])
ff = ForceField("openff_no_water-3.0.0-alpha0.offxml", 
                "opc3-1.0.1.offxml")

ic = ff.create_interchange(top)
ic.visualize()



/Users/jeffreywagner/micromamba/envs/off-tk-dev/lib/python3.12/site-packages/smirnoff99frosst/smirnoff99frosst.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


NGLWidget()

In [3]:
import openmm
from openff.units import Quantity, unit

simulation = ic.to_openmm_simulation(
    openmm.LangevinIntegrator(
        300 * openmm.unit.kelvin,
        1 / openmm.unit.picosecond,
        2 * openmm.unit.femtoseconds,
    )
)

# Energy minimize
simulation.minimizeEnergy()

# Add a reporter to record the structure every 1000 steps
dcd_reporter = openmm.app.DCDReporter("trajectory.dcd", 1000)
simulation.reporters.append(dcd_reporter)
simulation.context.setVelocitiesToTemperature(simulation.integrator.getTemperature())
simulation.runForClockTime(.1 * openmm.unit.minute)

In [3]:
import mdtraj
import nglview

trajectory: mdtraj.Trajectory = mdtraj.load(
    "trajectory.dcd", top=mdtraj.Topology.from_openmm(top.to_openmm())
)

view = nglview.show_mdtraj(trajectory.image_molecules())
view.add_representation("line", selection="water")
view.add_representation(
    "hyperball", radiusSize=1, radiusScale=0.5, selection="not protein and not water"
)
view

NGLWidget(max_frame=21)